# Brunn--Minkowski through an affine OT interpolation

This notebook generates `fig:gradflow-brunn-minkowski-ot`.  It gives a noise-free two-dimensional illustration of the determinant mechanism behind the Brunn--Minkowski proof.  For two ellipses, the quadratic Brenier map is affine, so
$$
    T_t=(1-t)\operatorname{Id}+tT
$$
transforms the red source ellipse into a family of ellipses.  The area is therefore computed exactly from a determinant, and the displayed curve compares
$$
    |T_t(A)|^{1/2}
    \quad\hbox{with}\quad
    (1-t)|A|^{1/2}+t|B|^{1/2}.
$$
No title is embedded in the PDFs; the LaTeX caption supplies the explanation.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Polygon
from IPython.display import Image, display

ROOT = Path.cwd()
if ROOT.name == "notebooks-figures":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "notebooks-figures"))

from figure_style import (
    BLUE,
    RED,
    VIOLET,
    GRAY,
    box_axes,
    figure_dir,
    interp_color,
    remove_axes,
    save_pdf,
    setup_matplotlib,
)

setup_matplotlib()
OUT = figure_dir("gradflow-brunn-minkowski-ot")
PNG_OUT = ROOT / "notebooks-figures" / "thumbnails"
PNG_OUT.mkdir(exist_ok=True)

rng = np.random.default_rng(20260619)

## Affine Brenier map between ellipses

We use centered ellipses with different anisotropy and orientation.  If the source support is
$A=m_0+A_0B_2$ and the target support is $B=m_1+A_1B_2$, the affine quadratic OT map has linear part
$$
M=\Sigma_0^{-1/2}\bigl(\Sigma_0^{1/2}\Sigma_1\Sigma_0^{1/2}\bigr)^{1/2}\Sigma_0^{-1/2},
\qquad \Sigma_i=A_iA_i^\top .
$$
The interpolated support is
$T_t(A)=m_t+((1-t)I+tM)A_0B_2$.

In [ ]:
def rot(theta):
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[c, -s], [s, c]])


def sym_sqrt(A):
    vals, vecs = np.linalg.eigh((A + A.T) / 2)
    return (vecs * np.sqrt(np.maximum(vals, 0.0))) @ vecs.T


def sym_invsqrt(A):
    vals, vecs = np.linalg.eigh((A + A.T) / 2)
    return (vecs * (1.0 / np.sqrt(np.maximum(vals, 1e-15)))) @ vecs.T


def ellipse_matrix(r1, r2, theta):
    return rot(theta) @ np.diag([r1, r2])

m0 = np.array([-0.70, -0.04])
m1 = np.array([0.82, 0.10])
A0 = ellipse_matrix(1.12, 0.42, 0.18)
A1 = ellipse_matrix(0.58, 1.08, -0.72)
S0 = A0 @ A0.T
S1 = A1 @ A1.T
M = sym_invsqrt(S0) @ sym_sqrt(sym_sqrt(S0) @ S1 @ sym_sqrt(S0)) @ sym_invsqrt(S0)

angles = np.linspace(0, 2 * np.pi, 420, endpoint=False)
unit_circle = np.column_stack([np.cos(angles), np.sin(angles)])

area0 = np.pi * abs(np.linalg.det(A0))
area1 = np.pi * abs(np.linalg.det(A1))
area_sqrt_bound = lambda t: (1 - t) * np.sqrt(area0) + t * np.sqrt(area1)

def ellipse_at(t):
    L = ((1.0 - t) * np.eye(2) + t * M) @ A0
    m = (1.0 - t) * m0 + t * m1
    pts = m + unit_circle @ L.T
    area = np.pi * abs(np.linalg.det(L))
    return m, L, pts, area

# A small transported grid makes the affine deformation readable without relying on noisy samples.
grid_lines = []
for u in np.linspace(-0.72, 0.72, 7):
    v = np.linspace(-np.sqrt(max(1 - u**2, 0)), np.sqrt(max(1 - u**2, 0)), 120)
    grid_lines.append(np.column_stack([np.full_like(v, u), v]))
for v in np.linspace(-0.72, 0.72, 7):
    u = np.linspace(-np.sqrt(max(1 - v**2, 0)), np.sqrt(max(1 - v**2, 0)), 120)
    grid_lines.append(np.column_stack([u, np.full_like(u, v)]))

def transported_line(line, t):
    source = m0 + line @ A0.T
    return ((1 - t) * np.eye(2) + t * M) @ (source - m0).T


## Exported panels

Five panels show the interpolated support, and one panel plots the square-root area.  The horizontal dashed segment in the curve panel is not a fit: it is the Brunn--Minkowski affine lower bound.

In [ ]:
times = np.array([0.0, 0.25, 0.5, 0.75, 1.0])

def set_tight_equal_limits(ax, pts, margin=0.18):
    """Crop each support tightly while keeping an equal aspect ratio."""
    lo = pts.min(axis=0)
    hi = pts.max(axis=0)
    center = 0.5 * (lo + hi)
    half = 0.5 * max(hi - lo) * (1.0 + margin)
    ax.set_xlim(center[0] - half, center[0] + half)
    ax.set_ylim(center[1] - half, center[1] + half)

for t in times:
    fig, ax = plt.subplots(figsize=(1.82, 1.70))
    color = interp_color(float(t))
    _, _, pts, _ = ellipse_at(float(t))
    ax.add_patch(Polygon(pts, closed=True, facecolor=color, edgecolor="none", alpha=0.23, zorder=1))
    ax.plot(pts[:, 0], pts[:, 1], color=color, lw=1.35, zorder=3)
    L_t = ((1 - t) * np.eye(2) + t * M) @ A0
    m_t = (1 - t) * m0 + t * m1
    for line in grid_lines:
        moved = m_t + line @ L_t.T
        ax.plot(moved[:, 0], moved[:, 1], color=color, lw=0.38, alpha=0.42, zorder=2)
    ax.set_aspect("equal")
    set_tight_equal_limits(ax, pts)
    remove_axes(ax)
    save_pdf(fig, OUT / f"ellipse-t{int(round(100*t)):03d}.pdf", pad_inches=0.025)
    plt.close(fig)

curve_t = np.linspace(0, 1, 301)
area_sqrt = np.array([np.sqrt(ellipse_at(float(t))[3]) for t in curve_t])
bound = np.array([area_sqrt_bound(float(t)) for t in curve_t])

fig, ax = plt.subplots(figsize=(2.72, 1.72))
ax.plot(curve_t, area_sqrt, color=VIOLET, lw=1.55, label=r"$|T_t(A)|^{1/2}$")
ax.plot(curve_t, bound, color=GRAY, lw=1.05, ls="--", label=r"linear bound")
ax.fill_between(curve_t, bound, area_sqrt, color=VIOLET, alpha=0.10, linewidth=0)
ax.scatter(times, [np.sqrt(ellipse_at(float(t))[3]) for t in times], s=12, c=[interp_color(float(t)) for t in times], zorder=5)
ax.set_xlim(0, 1)
ymin_curve = min(bound.min(), area_sqrt.min()) * 0.985
ymax_curve = max(bound.max(), area_sqrt.max()) * 1.018
ax.set_ylim(ymin_curve, ymax_curve)
ax.set_xlabel(r"$t$", labelpad=1.5)
ax.set_ylabel(r"square-root area", labelpad=2.0)
ax.tick_params(labelsize=7, pad=1.5)
ax.legend(loc="lower center", frameon=False, fontsize=7, ncol=2, handlelength=1.8, borderpad=0.1, labelspacing=0.15)
box_axes(ax)
save_pdf(fig, OUT / "area-concavity.pdf", pad_inches=0.035)
plt.close(fig)

# Compact PNG thumbnail matching the LaTeX layout.
thumb = plt.figure(figsize=(9.2, 1.55))
gs = thumb.add_gridspec(1, 6, width_ratios=[1, 1, 1, 1, 1, 2.15], wspace=0.34)
for k, t in enumerate(times):
    ax = thumb.add_subplot(gs[0, k])
    color = interp_color(float(t))
    _, _, pts, _ = ellipse_at(float(t))
    ax.add_patch(Polygon(pts, closed=True, facecolor=color, edgecolor="none", alpha=0.23, zorder=1))
    ax.plot(pts[:, 0], pts[:, 1], color=color, lw=1.15, zorder=3)
    L_t = ((1 - t) * np.eye(2) + t * M) @ A0
    m_t = (1 - t) * m0 + t * m1
    for line in grid_lines[::2]:
        moved = m_t + line @ L_t.T
        ax.plot(moved[:, 0], moved[:, 1], color=color, lw=0.30, alpha=0.36, zorder=2)
    ax.set_aspect("equal")
    set_tight_equal_limits(ax, pts)
    remove_axes(ax)
ax = thumb.add_subplot(gs[0, 5])
ax.plot(curve_t, area_sqrt, color=VIOLET, lw=1.45)
ax.plot(curve_t, bound, color=GRAY, lw=1.0, ls="--")
ax.fill_between(curve_t, bound, area_sqrt, color=VIOLET, alpha=0.10, linewidth=0)
ax.scatter(times, [np.sqrt(ellipse_at(float(t))[3]) for t in times], s=10, c=[interp_color(float(t)) for t in times], zorder=5)
ax.set_xlim(0, 1)
ax.set_ylim(ymin_curve, ymax_curve)
ax.set_xlabel(r"$t$", labelpad=1.0)
ax.set_ylabel("")
ax.tick_params(labelsize=6.5, pad=1.0)
box_axes(ax)
thumb.savefig(PNG_OUT / "gradflow-brunn-minkowski-ot.png", dpi=190, bbox_inches="tight", pad_inches=0.02)
plt.close(thumb)

print(f"saved panels in {OUT.relative_to(ROOT)}")

## Figure preview

The notebook preview shows the quantitative panel.  The LaTeX figure assembles this with the five geometric panels.

In [ ]:
display(Image(filename=str(PNG_OUT / "gradflow-brunn-minkowski-ot.png")))